In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
INPUT_PATH = 'Rental_Predictor_Data/zori_safmr_joined.csv'
OUTPUT_DIR = 'Rental_Predictor_Data'
 
TIER1_MIN_MONTHS = 60   # 5 years - required for trend model
TIER2_MIN_MONTHS = 36   # 3 years - sufficient for seasonality only
BEDROOMS = [0, 1, 2, 3, 4]

In [3]:
# Load the data
df = pd.read_csv(INPUT_PATH, low_memory=False, parse_dates=['date'])
df['zip_code'] = df['zip_code'].astype(str).str.zfill(5)
df = df.sort_values(['zip_code', 'date']).reset_index(drop=True)

In [4]:
# Filter to ZORI data
df_zori = df.dropna(subset=['zori']).copy()

In [5]:
# Assign tiers to the data
coverage = df_zori.groupby('zip_code').agg(
    months=('zori', 'count'),
    state=('State', 'first'),
    metro=('Metro', 'first'),
    county=('CountyName', 'first'),
    city=('City', 'first'),
).reset_index()

coverage['tier'] = np.where(
    coverage['months'] >= TIER1_MIN_MONTHS, 1,
    np.where(coverage['months'] >= TIER2_MIN_MONTHS, 2, 3)
)
# Building a fallback to metro data -> for each zip code, find it's metro and the metro median monthly ZORI
# used when a zip code lacks sufficient data
metro_by_month = (
    df_zori.groupby(['Metro', 'date'])['zori']
    .median()
    .reset_index()
    .rename(columns={'zori': 'metro_zori'})
)
    

In [6]:
# Engineer lag features to be used with XGBoost
tier1_zip_codes = coverage[coverage['tier'] == 1]['zip_code'].tolist()
df_t1 = df_zori[df_zori['zip_code'].isin(tier1_zip_codes)].copy()
df_t1 = df_t1.sort_values(['zip_code', 'date']).reset_index(drop=True)
# ZORI value grouped by zip code
grp = df_t1.groupby('zip_code')['zori']
# Create lag features
df_t1['zori_lag_1m']  = grp.shift(1) # Don't include current month's data to avoid leakage
df_t1['zori_lag_3m']  = grp.shift(3)
df_t1['zori_lag_6m']  = grp.shift(6)
df_t1['zori_lag_12m'] = grp.shift(12)
df_t1['zori_lag_24m'] = grp.shift(24)
df_t1['zori_lag_36m'] = grp.shift(36)

# Recent trend stats
df_t1['zori_roll_mean_6m']  = grp.transform(lambda x: x.shift(1).rolling(6).mean())
df_t1['zori_roll_mean_12m'] = grp.transform(lambda x: x.shift(1).rolling(12).mean())
df_t1['zori_roll_std_6m']   = grp.transform(lambda x: x.shift(1).rolling(6).std())

# Adding to replace year in numerical features
national_monthly = df_t1.groupby('date')['zori'].median().reset_index()
national_monthly.columns = ['date', 'national_median_zori']
df_t1 = df_t1.merge(national_monthly, on='date', how='left')
df_t1['zori_vs_national'] = df_t1['zori'] / df_t1['national_median_zori']

# Rate of change
df_t1['zori_mom_pct'] = (df_t1['zori'] - df_t1['zori_lag_1m']) / df_t1['zori_lag_1m'] * 100
df_t1['zori_yoy_pct'] = (df_t1['zori'] - df_t1['zori_lag_12m']) / df_t1['zori_lag_12m'] * 100
df_t1['zori_2y_pct']  = (df_t1['zori'] - df_t1['zori_lag_24m']) / df_t1['zori_lag_24m'] * 100

In [7]:
# SAFMR is annual - compute YoY % change per ZIP per bedroom
safmr_cols = [f'fmr_{br}br' for br in BEDROOMS]

# Get one SAFMR row per ZIP per fiscal year (deduplicate - already done but be safe)
safmr_annual = (
    df_t1.dropna(subset=['fmr_1br'])
    [['zip_code', 'fiscal_year'] + safmr_cols]
    .drop_duplicates(subset=['zip_code', 'fiscal_year'])
    .sort_values(['zip_code', 'fiscal_year'])
)

for br in BEDROOMS:
    col = f'fmr_{br}br'
    lag_col = f'fmr_{br}br_lag1y'
    yoy_col = f'fmr_{br}br_yoy_pct'
    safmr_annual[lag_col] = safmr_annual.groupby('zip_code')[col].shift(1)
    safmr_annual[yoy_col] = (
        (safmr_annual[col] - safmr_annual[lag_col]) / safmr_annual[lag_col] * 100
    )

# Merge the SAFMR annual features back onto monthly ZORI rows
df_t1 = df_t1.merge(
    safmr_annual[['zip_code', 'fiscal_year'] +
                 [f'fmr_{br}br_lag1y' for br in BEDROOMS] +
                 [f'fmr_{br}br_yoy_pct' for br in BEDROOMS]],
    on=['zip_code', 'fiscal_year'],
    how='left'
)

In [8]:
# Add in calendar features
df_t1['month']      = df_t1['date'].dt.month
df_t1['quarter']    = df_t1['date'].dt.quarter
df_t1['year']       = df_t1['date'].dt.year
df_t1['month_sin']  = np.sin(2 * np.pi * df_t1['month'] / 12)
df_t1['month_cos']  = np.cos(2 * np.pi * df_t1['month'] / 12)

In [9]:
# Build target values
df_t1['target_zori_12m'] = df_t1.groupby('zip_code')['zori'].shift(-12)
df_t1['target_yoy_pct']  = (
    (df_t1['target_zori_12m'] - df_t1['zori']) / df_t1['zori'] * 100
)
 
# Per-bedroom targets using ratio projection
for br in BEDROOMS:
    ratio_col = f'ratio_{br}br_to_2br'
    df_t1[f'target_fmr_{br}br_yoy_pct'] = df_t1.groupby('zip_code')[f'fmr_{br}br'].shift(-1)
    df_t1[f'target_fmr_{br}br_yoy_pct'] = (
        (df_t1[f'target_fmr_{br}br_yoy_pct'] - df_t1[f'fmr_{br}br']) /
        df_t1[f'fmr_{br}br'] * 100
    )

In [10]:
# Check and dropping rows with insufficient lag data
critical_features = ['zori_lag_12m', 'zori_lag_24m', 'zori_yoy_pct', 'target_yoy_pct']
before = len(df_t1)
df_t1 = df_t1.dropna(subset=critical_features)

In [11]:
# Seasonality
tier12_zips = coverage[coverage['tier'].isin([1, 2])]['zip_code'].tolist()
df_t12 = df_zori[df_zori['zip_code'].isin(tier12_zips)].copy()
df_t12['month'] = df_t12['date'].dt.month
df_t12['year']  = df_t12['date'].dt.year
 
# Annual mean per ZIP for each year
df_t12['annual_mean'] = df_t12.groupby(['zip_code', 'year'])['zori'].transform('mean')
df_t12['monthly_dev_pct'] = (df_t12['zori'] - df_t12['annual_mean']) / df_t12['annual_mean'] * 100
 
seasonal = (
    df_t12.groupby(['zip_code', 'month'])['monthly_dev_pct']
    .agg(avg_dev_pct='mean', std_dev_pct='std', obs='count')
    .reset_index()
    .round(4)
)
 
# Flag reliability: low obs == less reliable
seasonal['reliable'] = seasonal['obs'] >= 3

In [12]:
# Metro level seasonality
df_zori['month'] = df_zori['date'].dt.month
df_zori['year']  = df_zori['date'].dt.year
df_zori['annual_mean'] = df_zori.groupby(['zip_code', 'year'])['zori'].transform('mean')
df_zori['monthly_dev_pct'] = (df_zori['zori'] - df_zori['annual_mean']) / df_zori['annual_mean'] * 100
 
metro_seasonal = (
    df_zori.dropna(subset=['Metro'])
    .groupby(['Metro', 'month'])['monthly_dev_pct']
    .agg(avg_dev_pct='mean', std_dev_pct='std', obs='count')
    .reset_index()
    .round(4)
)

In [17]:
# Save the outputs

# Trend features
trend_cols = (
    ['zip_code', 'State', 'Metro', 'CountyName', 'date', 'month', 'quarter', 'year',
     'month_sin', 'month_cos', 'fiscal_year',
     'zori', 'zori_sa', 'zori_lag_1m', 'zori_lag_3m', 'zori_lag_6m',
     'zori_lag_12m', 'zori_lag_24m', 'zori_lag_36m',
     'zori_roll_mean_6m', 'zori_roll_mean_12m', 'zori_roll_std_6m',
     'zori_mom_pct', 'zori_yoy_pct', 'zori_2y_pct'] +
    [f'fmr_{br}br' for br in BEDROOMS] +
    [f'fmr_{br}br_lag1y' for br in BEDROOMS] +
    [f'fmr_{br}br_yoy_pct' for br in BEDROOMS] +
    [f'ratio_{br}br_to_2br' for br in BEDROOMS] +
    ['target_yoy_pct'] +
    [f'target_fmr_{br}br_yoy_pct' for br in BEDROOMS]
)
trend_cols = [c for c in trend_cols if c in df_t1.columns]
df_t1[trend_cols].to_csv(f'{OUTPUT_DIR}/features_trend.csv', index=False)
print(f"  features_trend.csv: {len(df_t1):,} rows, {len(trend_cols)} features")
 
# Seasonality profiles
seasonal.to_csv(f'{OUTPUT_DIR}/features_seasonal.csv', index=False)
print(f"  features_seasonal.csv: {len(seasonal):,} rows")
 
# Metro seasonality fallback
metro_seasonal.to_csv(f'{OUTPUT_DIR}/features_metro_seasonal.csv', index=False)
print(f"  features_metro_seasonal.csv: {len(metro_seasonal):,} rows")
 
# Tier map
tier_map = coverage[['zip_code', 'tier', 'state', 'metro', 'county', 'city', 'months']].copy()
tier_map.to_csv(f'{OUTPUT_DIR}/zip_tier_map.csv', index=False)


  features_trend.csv: 196,215 rows, 51 features
  features_seasonal.csv: 57,564 rows
  features_metro_seasonal.csv: 6,662 rows


In [14]:
print("\n=== PIPELINE COMPLETE ===")
print(f"Trend training rows:        {len(df_t1):,}")
print(f"Trend features:             {len(trend_cols)}")
print(f"Seasonal ZIPs (Tier 1+2):  {seasonal['zip_code'].nunique():,}")
print(f"Metro fallback metros:      {metro_seasonal['Metro'].nunique():,}")
print(f"\nTarget variable stats:")
print(df_t1['target_yoy_pct'].describe().round(3))
 
print("\nSample - Carlsbad (92008) last 3 rows:")
sample = df_t1[df_t1['zip_code'] == '92008'].tail(3)
print(sample[['zip_code','date','zori','zori_yoy_pct','fmr_1br','target_yoy_pct']].to_string())


=== PIPELINE COMPLETE ===
Trend training rows:        196,215
Trend features:             51
Seasonal ZIPs (Tier 1+2):  4,797
Metro fallback metros:      674

Target variable stats:
count    196215.000
mean          5.098
std           5.860
min         -25.398
25%           1.628
50%           3.940
75%           7.225
max          50.026
Name: target_yoy_pct, dtype: float64

Sample - Carlsbad (92008) last 3 rows:
       zip_code       date         zori  zori_yoy_pct  fmr_1br  target_yoy_pct
249548    92008 2025-02-28  2982.684323      2.886199   2590.0        3.246054
249549    92008 2025-03-31  3004.284936      2.780684   2590.0        3.844307
249550    92008 2025-04-30  3021.959682      3.810023   2590.0        5.141356


In [18]:
# COVID era analysis
df = pd.read_csv('Rental_Predictor_Data/features_trend.csv')
print(df['target_yoy_pct'].quantile([0.95, 0.99, 0.999])) # Upper clip ~25.4%
print(df['target_yoy_pct'].quantile([0.001, 0.01, 0.05])) # Lower clip ~15%

0.950    16.672051
0.990    25.358726
0.999    35.027660
Name: target_yoy_pct, dtype: float64
0.001   -15.046979
0.010    -5.715090
0.050    -1.862634
Name: target_yoy_pct, dtype: float64


In [19]:
# Considering clipping at the 99th percentile to account for COVID era surges (better for generalization)
# Lower tail seems better behaved